# Promotion Effectiveness Dashboard
**Marketing Analytics Module 2** — SCAN*PRO log-log model · Incremental sales measurement

## The question

Does featuring a product on the e-commerce platform's main page actually boost sales? And if so, by how much?

## Why we can't just compare promoted vs non-promoted weeks

A naïve approach would be: compute average sales during promoted weeks, subtract average sales during non-promoted weeks, and call the difference the "promotion effect." This is unreliable because:

1. **Price confounding:** Retailers often discount products *and* feature them simultaneously. Without controlling for price, the "promotion effect" is really "promotion + discount effect" bundled together.
2. **Seasonal confounding:** Promotions tend to cluster around high-demand periods (holidays, back-to-school). A product featured in December sells more than in February — but that's seasonality, not promotion effectiveness.
3. **Trend confounding:** If demand is growing over time and promotions happen more often in later weeks, the naïve comparison attributes the growth trend to promotion.

We need a model that **controls for these confounders** to isolate the *pure* promotion effect.

## Why SCAN*PRO

The SCAN\*PRO model (Van Heerde et al., 2002) was specifically designed for this problem. It is the standard model in marketing science for measuring promotion effectiveness from observational sales data. We chose it because:

- It separates the promotion effect (β₂) from the price effect (β₁) by including both in the same regression
- It controls for dynamic pricing effects via lagged prices — a feature shown by the original authors to reduce bias in the promotion coefficient
- It includes trend and seasonal controls
- It uses OLS, which is simple, interpretable, and appropriate when the goal is **coefficient estimation** (measuring β₂) rather than prediction

**Alternatives considered:**
- *Raw comparison / t-test:* Confounded by price, seasonality, and trend (see above).
- *Difference-in-differences / synthetic control:* Would require a suitable untreated control group. All SKUs on this platform can potentially be promoted, so there is no clean control.
- *Bayesian structural time series:* More sophisticated but similar results for this sample size (~98 weeks), with much higher implementation complexity.

## Model specification

$$\ln(\text{sales}_{it}) = \alpha + \beta_1 \cdot \ln(\text{price}_{it}) + \beta_2 \cdot \text{feat}_{it} + \beta_3 \cdot \text{trend}_t + \beta_4 \cdot \ln(\text{price}_{i,t-1}) + \beta_5 \cdot \ln(\text{price}_{i,t-2}) + \sum_k \gamma_k \cdot \text{month}_k + \varepsilon_{it}$$

### Why this functional form?

**Log-log for price variables:** Taking the log of both sales and price means that β₁ directly equals the price elasticity (% change in demand per 1% change in price). This is a standard result in microeconomics. The same applies to β₄ and β₅ for lagged prices.

**Linear (not logged) promotion dummy:** `feat_main_page` is binary (0/1) — you can't take the log of zero. Entering it linearly in a log-dependent model gives a *semi-elasticity*: exp(β₂) − 1 = proportional sales lift when featured. This is the standard SCAN\*PRO parameterisation.

**Why OLS?** The goal is to estimate β₂ (the promotion effect), not to predict sales. OLS gives unbiased, interpretable coefficient estimates with standard errors and p-values for hypothesis testing. With ~98 observations per SKU and 15–17 parameters, OLS has sufficient power.

### Why each feature is in the model

| Feature | Role | What happens if we omit it |
|---------|------|---------------------------|
| **ln(price_t)** | Controls for contemporaneous price | β₂ is biased upward (captures discount + promo together) |
| **feat_main_page** | The variable of interest | — |
| **trend** | Controls for secular demand growth/decline | β₂ is biased if promotions cluster in later (higher-demand) weeks |
| **ln(price_{t-1}), ln(price_{t-2})** | Controls for reference-price effects and stockpiling | β₁ and β₂ both biased (Van Heerde et al., 2002). Consumers anchor to recent prices; a past discount can cannibalise current demand |
| **month dummies** | Controls for seasonal demand patterns | β₂ is biased if promotions overlap with seasonal peaks (e.g. holiday periods) |

### Interpreting the output

- **Lift** = exp(β₂) − 1. If β₂ = 0.25, lift = exp(0.25) − 1 = 28.4% → sales increase by 28.4% during promo weeks.
- **Incremental sales/week** = baseline sales × lift. If the SKU normally sells 100 units/week and lift is 28.4%, each promo week generates ~28 extra units.
- **Total incremental** = incremental/week × number of promoted weeks.
- **Significance (p < 0.05)** means we can be 95% confident the promotion effect is not zero.

**Reference:** Van Heerde, H.J., Leeflang, P.S.H., & Wittink, D.R. (2002). How promotions work: SCAN\*PRO-based evolutionary model building. *Schmalenbach Business Review*, 54, 198–220.

In [10]:
# Load libraries for data handling, interactive charts, and the SCAN*PRO regression model
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import statsmodels.api as sm
from IPython.display import display, HTML
import ipywidgets as widgets
import warnings
warnings.filterwarnings("ignore")

pio.templates.default = "plotly_white"

C_GREEN = "#059669"
C_INDIGO = "#6366f1"

# Helper functions to display styled KPI summary boxes
def metric_card(label, value, sub=""):
    return f'''<div style="flex:1;min-width:155px;background:#f8fafc;border:1px solid #e2e8f0;
    border-radius:12px;padding:1.1rem 1.4rem;">
    <div style="font-size:0.72rem;color:#64748b;text-transform:uppercase;letter-spacing:0.05em;font-weight:500;">{label}</div>
    <div style="font-size:1.45rem;font-weight:700;color:#0f172a;margin-top:0.2rem;font-family:monospace;">{value}</div>
    <div style="font-size:0.78rem;color:#64748b;margin-top:0.15rem;">{sub}</div></div>'''

def metric_row(cards):
    return HTML(f'<div style="display:flex;gap:1rem;margin:1rem 0;flex-wrap:wrap;">{"".join(cards)}</div>')

print("Imports loaded")


Imports loaded


In [ ]:
# Read raw and processed sales data, then build a quick lookup of each SKU's category, color, and avg price
raw = pd.read_csv("../data/data_raw.csv")
proc = pd.read_csv("../data/data_processed.csv")
raw["week"] = pd.to_datetime(raw["week"])
proc["week"] = pd.to_datetime(proc["week"])
raw["feat_main_page"] = raw["feat_main_page"].astype(str).str.lower().eq("true").astype(int)

sku_meta = raw.groupby("sku").agg(
    functionality=("functionality", "first"),
    color=("color", "first"),
    avg_price=("price", "mean"),
).reset_index()

print(f"Loaded {raw.sku.nunique()} SKUs, {raw.week.nunique()} weeks")
print(f"Overall promo rate: {raw.feat_main_page.mean():.1%}")


Loaded 44 SKUs, 100 weeks
Overall promo rate: 35.8%


## Building the model specification

This section explains **how** we arrived at the model specification above — every design choice, what alternatives we considered, and why.

### Estimation vs prediction: the fundamental distinction

In the Demand Forecasting module, the goal was **prediction** — minimise forecast error. That meant we aggressively reduced features (20+ → 5) to avoid overfitting, because every unnecessary feature costs prediction accuracy on unseen data.

Here the goal is fundamentally different: **coefficient estimation**. We need an accurate, unbiased estimate of β₂ (the promotion effect). The risk is not overfitting — it's **omitted variable bias**. If we leave out a relevant control variable, its effect gets absorbed into β₂, making our promotion estimate wrong. So the logic flips:

| | Demand Forecasting | Promotion Effectiveness |
|---|---|---|
| **Goal** | Predict future sales (minimise MAE) | Estimate β₂ (promotion lift) |
| **Risk** | Overfitting → poor generalisation | Omitted variable bias → wrong β₂ |
| **Feature strategy** | Fewer is better | Include all relevant controls |
| **Method** | ML models (RF, MLP) | OLS (interpretable coefficients) |
| **Lagged prices** | Removed (multicollinear, hurts prediction) | Kept (omitting them biases β₁ and β₂) |
| **Month dummies** | Replaced with 1 quarter variable | Kept (11 dummies control seasonal confounding) |

### Step 1: Why log(sales) as the dependent variable?

Three reasons:
1. **Multiplicative relationships become additive.** A promotion doesn't add a fixed 50 units to every SKU — it multiplies demand by some factor (e.g. ×1.3). Log-transforming sales turns this multiplicative relationship into an additive one that OLS can handle.
2. **Heteroscedasticity reduction.** High-demand SKUs have larger absolute fluctuations. Logging compresses the scale and stabilises the variance, satisfying OLS assumptions.
3. **Percentage interpretation.** In a log-linear model, coefficients naturally interpret as approximate percentage effects, which is what business stakeholders want to know.

### Step 2: Why log(price) but NOT log(feat_main_page)?

**Price is continuous.** Taking its log means β₁ = ∂ln(Q)/∂ln(P) = price elasticity (% change in demand per 1% change in price). This is a standard microeconomic result.

**feat_main_page is binary (0 or 1).** You cannot take log(0) — it's undefined. So the promotion dummy enters linearly. In a log-dependent model, a linear binary variable gives a *semi-elasticity*: the coefficient β₂ means that switching from 0 to 1 changes log(sales) by β₂, which translates to a proportional change of exp(β₂) − 1 in sales levels. This is the standard parameterisation in marketing science (Van Heerde et al., 2002).

### Step 3: Why include ln(price_t) at all?

Because promotions and price discounts often happen simultaneously — a retailer features a product on the main page *and* cuts its price. If we omit price from the model, β₂ captures both effects: "being featured" + "being discounted." This overstates the promotion's own contribution. Including ln(price_t) separates the two effects, giving us the *pure* promotion lift after controlling for price.

### Step 4: Why lagged prices ln(price_{t-1}) and ln(price_{t-2})?

This is the defining feature of the SCAN\*PRO model and the most counterintuitive design choice. We removed lagged prices from demand forecasting because they were multicollinear with current price and hurt prediction. **Why keep them here?**

The answer is omitted variable bias. Van Heerde et al. (2002) showed empirically that omitting price lags from the promotion model systematically biases *both* the price coefficient (β₁) and the promotion coefficient (β₂). The economic mechanisms are:

- **Reference-price formation:** Consumers anchor to recent prices. If last week's price was £10 and this week it's £8, the £2 gap feels like a deal — even if £8 is the normal price. This reference-price effect influences current demand beyond what current price alone captures.
- **Stockpiling:** A price cut in week t−1 may cause consumers to buy extra and stockpile, reducing demand in week t even if the current price hasn't changed. Without lagged prices to absorb this effect, it leaks into β₂.

**Multicollinearity is acceptable here** because we're not trying to predict — we're trying to get unbiased coefficient estimates. High VIF inflates standard errors (wider confidence intervals) but does not bias the point estimates. This is a conscious trade-off: we accept less precise estimates in exchange for less biased ones.

### Step 5: Why trend?

If demand is growing over time (e.g. because the product category is becoming popular) and promotions tend to happen more often in later weeks, the model would mistakenly attribute part of the growth trend to promotions. The linear trend variable absorbs this secular growth/decline, preventing it from contaminating β₂.

### Step 6: Why 11 month dummies (not 1 quarter variable like in demand forecasting)?

In demand forecasting, we replaced 11 month dummies with 1 quarter variable to reduce features (prediction → fewer is better). Here the logic is different:

- Promotions may cluster in specific months (e.g. November for Black Friday, December for Christmas). If we use a coarse quarterly variable, within-quarter monthly variation in both promotions and demand would confound β₂.
- With ~98 observations and ~16 parameters (intercept + 5 regressors + 10 month dummies), we have ~82 degrees of freedom — sufficient for OLS.
- The cost of including month dummies (wider standard errors from fewer degrees of freedom) is less than the cost of omitting them (biased β₂ from seasonal confounding).

### Step 7: Why OLS and not something more sophisticated?

The goal is to estimate one number — β₂ — and test whether it's statistically different from zero. OLS gives:
- Unbiased point estimates (under the standard linear regression assumptions)
- Valid standard errors for hypothesis testing and confidence intervals
- Full interpretability — every coefficient has a direct economic meaning
- Simplicity — the results are easy to explain to a marketing manager

More complex alternatives (Bayesian, machine learning, causal inference methods) would require stronger assumptions, more data, or more complexity without materially improving the answer to "does featuring this product boost sales?" for this dataset size.

### Summary: the complete model

$$\ln(\text{sales}_{it}) = \underbrace{\alpha}_{\text{base level}} + \underbrace{\beta_1 \cdot \ln(\text{price}_{it})}_{\text{price effect}} + \underbrace{\beta_2 \cdot \text{feat}_{it}}_{\text{PROMOTION (target)}} + \underbrace{\beta_3 \cdot \text{trend}_t}_{\text{growth control}} + \underbrace{\beta_4 \cdot \ln(\text{price}_{i,t-1}) + \beta_5 \cdot \ln(\text{price}_{i,t-2})}_{\text{reference-price / stockpiling}} + \underbrace{\sum_k \gamma_k \cdot \text{month}_k}_{\text{seasonal control}} + \varepsilon_{it}$$

Every variable except `feat_main_page` is a **control** — its purpose is to make β₂ as clean an estimate of the true promotion effect as possible.

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# SCAN*PRO MODEL IMPLEMENTATION
# ══════════════════════════════════════════════════════════════════════════════
# This code implements the model specification justified in the cell above.
# It is fitted per-SKU via OLS because:
#   1. Different SKUs have different price elasticities and promo responses
#   2. Per-SKU fitting allows each SKU to have its own β₁, β₂, etc.
#   3. With ~98 obs and ~16 params per SKU, OLS has adequate degrees of freedom

def run_scanpro_all(proc):
    """Fit SCAN*PRO log-log OLS for every SKU. Returns DataFrame of results.
    
    Model: ln(sales) = α + β₁·ln(price) + β₂·feat + β₃·trend 
                      + β₄·ln(price_lag1) + β₅·ln(price_lag2) + Σγ·month + ε
    
    The target coefficient is β₂ (promotion effect).
    All other variables are controls to prevent omitted variable bias on β₂.
    """
    results = []

    for sku_id in sorted(proc['sku'].unique()):
        df = proc[proc['sku'] == sku_id].sort_values('week').copy()
        df = df[df['weekly_sales'] > 0]  # log(0) is undefined

        # Skip SKUs with too few observations or no promotion variation
        # (need both promoted and non-promoted weeks to estimate β₂)
        if len(df) < 15 or df['feat_main_page'].nunique() < 2:
            continue

        # ── Dependent variable: log(sales) ──
        # Log transform because: (1) promotion effects are multiplicative,
        # (2) reduces heteroscedasticity, (3) coefficients ≈ % effects
        y = np.log(df['weekly_sales'].values)

        # ── Build the regressor matrix ──
        X = pd.DataFrame(index=df.index)

        # log(price): controls for contemporaneous price so β₂ captures
        # PURE promo effect, not promo + discount bundled together.
        # Log form → β₁ = price elasticity directly.
        X['log_price'] = np.log(df['price'].values)

        # feat_main_page: THE VARIABLE OF INTEREST.
        # Binary (0/1), enters linearly (can't log a binary).
        # exp(β₂) - 1 = proportional sales lift when featured.
        X['feat_main_page'] = df['feat_main_page'].values

        # trend: absorbs secular demand growth/decline so β₂ isn't biased
        # by promotions happening to coincide with later (higher-demand) weeks.
        X['trend'] = df['trend'].values

        # Lagged log prices: the defining feature of SCAN*PRO.
        # Capture reference-price formation (consumers anchor to recent prices)
        # and stockpiling (past discounts cannibalise current demand).
        # Van Heerde et al. (2002) showed omitting these biases β₁ AND β₂.
        # YES, they're collinear with current price — but in estimation problems
        # (unlike prediction problems), multicollinearity widens CIs but doesn't
        # bias point estimates. We accept wider CIs for less biased β₂.
        if 'price-1' in df.columns:
            p1 = df['price-1'].values
            p1 = np.where(p1 > 0, p1, df['price'].values)
            X['log_price_lag1'] = np.log(p1)
        if 'price-2' in df.columns:
            p2 = df['price-2'].values
            p2 = np.where(p2 > 0, p2, df['price'].values)
            X['log_price_lag2'] = np.log(p2)

        # Month dummies: absorb seasonal demand patterns.
        # We use 11 dummies (not 1 quarter) because promotions may cluster in
        # specific months. Coarser seasonal controls would leave within-quarter
        # variation that confounds β₂. With ~98 obs and ~16 params, we have
        # ~82 degrees of freedom — adequate for OLS.
        month_cols = [c for c in df.columns if c.startswith('month_')]
        for c in month_cols:
            X[c] = df[c].values

        X = sm.add_constant(X)
        try:
            model = sm.OLS(y, X).fit(cov_type='HC1')  # Robust SE for heteroscedasticity
        except Exception:
            continue

        # ── Extract promotion metrics ──
        feat_coef = model.params.get('feat_main_page', 0)
        feat_pval = model.pvalues.get('feat_main_page', 1)
        feat_se = model.bse.get('feat_main_page', 0)
        price_coef = model.params.get('log_price', 0)

        promo_weeks = int(df['feat_main_page'].sum())
        avg_sales_off = df.loc[df['feat_main_page'] == 0, 'weekly_sales'].mean() if (df['feat_main_page'] == 0).any() else df['weekly_sales'].mean()
        avg_sales_on = df.loc[df['feat_main_page'] == 1, 'weekly_sales'].mean() if promo_weeks > 0 else 0

        # Lift = exp(β₂) - 1: proportional increase in sales during promo weeks
        pct_lift = (np.exp(feat_coef) - 1) * 100
        # Incremental = baseline_sales × lift: extra units per promo week
        incr_per_week = avg_sales_off * (np.exp(feat_coef) - 1)
        # 95% CI via delta method: exp(β₂ ± 1.96·SE) - 1
        ci_lo = (np.exp(feat_coef - 1.96 * feat_se) - 1) * 100
        ci_hi = (np.exp(feat_coef + 1.96 * feat_se) - 1) * 100

        results.append({
            'sku': sku_id, 'n_obs': len(df), 'r_squared': model.rsquared,
            'feat_coef': feat_coef, 'feat_se': feat_se, 'feat_pval': feat_pval,
            'price_elasticity': price_coef,
            'pct_lift': pct_lift, 'lift_ci_lo': ci_lo, 'lift_ci_hi': ci_hi,
            'avg_sales': df['weekly_sales'].mean(),
            'avg_sales_on': avg_sales_on, 'avg_sales_off': avg_sales_off,
            'promo_weeks': promo_weeks,
            'incr_per_week': incr_per_week,
            'total_incremental': incr_per_week * promo_weeks,
            'model': model,
        })

    return pd.DataFrame(results)


def run_scanpro_single(proc, sku_id):
    """Full OLS model for one SKU — used in the deep-dive for diagnostics."""
    df = proc[proc['sku'] == sku_id].sort_values('week').copy()
    df = df[df['weekly_sales'] > 0]
    y = np.log(df['weekly_sales'].values)

    X = pd.DataFrame(index=df.index)
    X['log_price'] = np.log(df['price'].values)
    X['feat_main_page'] = df['feat_main_page'].values
    X['trend'] = df['trend'].values
    if 'price-1' in df.columns:
        p1 = df['price-1'].values; p1 = np.where(p1 > 0, p1, df['price'].values)
        X['log_price_lag1'] = np.log(p1)
    if 'price-2' in df.columns:
        p2 = df['price-2'].values; p2 = np.where(p2 > 0, p2, df['price'].values)
        X['log_price_lag2'] = np.log(p2)
    month_cols = [c for c in df.columns if c.startswith('month_')]
    for c in month_cols:
        X[c] = df[c].values
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit(cov_type='HC1')  # Robust SE for heteroscedasticity
    return model, df

print('SCAN*PRO functions defined')


SCAN*PRO functions defined


In [13]:
# Run the promotion model on every SKU and show summary KPIs (how many were significant, median lift, etc.)
res_df = run_scanpro_all(proc)

n_sig_05 = (res_df["feat_pval"] < 0.05).sum()
n_sig_10 = (res_df["feat_pval"] < 0.10).sum()
median_lift = res_df.loc[res_df["feat_pval"] < 0.05, "pct_lift"].median()
total_incr = res_df.loc[res_df["feat_pval"] < 0.05, "total_incremental"].sum()

display(metric_row([
    metric_card("SKUs modelled", f"{len(res_df)}", "with sufficient data"),
    metric_card("Significant (p<0.05)", f"{n_sig_05} / {len(res_df)}", f"also {n_sig_10} at p<0.10"),
    metric_card("Median lift (sig.)", f"{median_lift:.1f}%", "SCAN*PRO exp(β)−1"),
    metric_card("Total incremental", f"{total_incr:,.0f}", "units across sig. SKUs"),
    metric_card("Avg model R²", f"{res_df.r_squared.mean():.3f}", "across all SKUs"),
]))


## Portfolio overview

In [14]:
# Horizontal bar chart: top 20 SKUs ranked by promotion lift (%), with 95% confidence intervals
df_plot = res_df.merge(sku_meta[["sku", "functionality"]], on="sku", how="left")
df_plot["label"] = df_plot.apply(lambda r: f"SKU {int(r.sku)} · {r.functionality[:22]}", axis=1)
df_plot = df_plot.sort_values("pct_lift", ascending=True).tail(20)

sig_cat = ["sig" if p < 0.05 else ("marg" if p < 0.10 else "ns") for p in df_plot["feat_pval"]]
fill_map  = {"sig": "#059669", "marg": "#f59e0b", "ns": "#cbd5e1"}
border_map = {"sig": "#047857", "marg": "#d97706", "ns": "#94a3b8"}
colors  = [fill_map[s] for s in sig_cat]
borders = [border_map[s] for s in sig_cat]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=df_plot["label"], x=df_plot["pct_lift"], orientation="h",
    marker=dict(color=colors, line=dict(width=1, color=borders)),
    error_x=dict(type="data", symmetric=False,
                 array=(df_plot["lift_ci_hi"] - df_plot["pct_lift"]).values,
                 arrayminus=(df_plot["pct_lift"] - df_plot["lift_ci_lo"]).values,
                 color="#64748b", thickness=1.5, width=4),
    text=[f"  {v:.0f}%" for v in df_plot["pct_lift"]], textposition="outside",
    textfont=dict(size=11, family="monospace", color="#1e293b"),
    customdata=np.column_stack([df_plot["lift_ci_lo"], df_plot["lift_ci_hi"], df_plot["feat_pval"]]),
    hovertemplate="<b>%{y}</b><br>Lift: %{x:.1f}%<br>"
                  "CI: [%{customdata[0]:.1f}%, %{customdata[1]:.1f}%]<br>"
                  "p = %{customdata[2]:.4f}<extra></extra>",
))

fig.add_shape(type="line", x0=0, x1=0, y0=-0.5, y1=len(df_plot) - 0.5,
              line=dict(width=2, color="#334155"))

for txt, col, yp in [("<b>●</b> p < 0.05", "#059669", 1.08),
                      ("<b>●</b> p < 0.10", "#f59e0b", 1.04),
                      ("<b>●</b> not sig.", "#94a3b8", 1.00)]:
    fig.add_annotation(text=txt, x=1.0, y=yp, xref="paper", yref="paper",
                       showarrow=False, font=dict(size=11, color=col), xanchor="right")

fig.update_layout(
    title=dict(
        text="<b>SCAN*PRO Promotion Lift by SKU</b><br>"
             "<span style='font-size:12px;color:#64748b'>Top 20 — estimated lift (%) with 95% confidence intervals</span>",
        font=dict(size=16, color="#0f172a"), x=0.01, xanchor="left"),
    height=620, margin=dict(l=260, r=80, t=95, b=50),
    xaxis=dict(title=dict(text="Promotion lift (%)", font=dict(size=12, color="#475569")),
               gridcolor="#f1f5f9", gridwidth=1, zeroline=False, dtick=100,
               tickfont=dict(size=10, color="#64748b")),
    yaxis=dict(tickfont=dict(size=10.5, color="#334155")),
    plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
    hoverlabel=dict(bgcolor="white", bordercolor="#cbd5e1", font_size=12),
    bargap=0.22,
)
fig.show()


In [15]:
# Bar chart: total extra units sold thanks to promotions, for statistically significant SKUs only
df_inc = res_df[res_df["feat_pval"] < 0.10].copy()
df_inc = df_inc.merge(sku_meta[["sku", "functionality"]], on="sku", how="left")
df_inc["label"] = df_inc.apply(lambda r: f"SKU {int(r.sku)} · {r.functionality[:18]}", axis=1)
df_inc = df_inc.sort_values("total_incremental", ascending=True).tail(15)

fig = go.Figure(go.Bar(
    y=df_inc["label"], x=df_inc["total_incremental"], orientation="h",
    marker=dict(
        color=df_inc["total_incremental"],
        colorscale=[[0, "#d1fae5"], [0.3, "#6ee7b7"], [0.6, "#10b981"], [1.0, "#065f46"]],
        line=dict(width=0.8, color="#047857"),
    ),
    text=[f"  {v:,.0f}" for v in df_inc["total_incremental"]], textposition="outside",
    textfont=dict(size=11, family="monospace", color="#1e293b"),
    customdata=np.column_stack([df_inc["pct_lift"], df_inc["promo_weeks"], df_inc["feat_pval"]]),
    hovertemplate="<b>%{y}</b><br>Incremental: %{x:,.0f} units<br>"
                  "Lift: %{customdata[0]:.1f}%<br>Promo weeks: %{customdata[1]:.0f}<br>"
                  "p = %{customdata[2]:.4f}<extra></extra>",
))

fig.update_layout(
    title=dict(
        text="<b>Total Incremental Sales from Promotions</b><br>"
             "<span style='font-size:12px;color:#64748b'>Significant SKUs (p < 0.10) — cumulative units across all promoted weeks</span>",
        font=dict(size=16, color="#0f172a"), x=0.01, xanchor="left"),
    height=500, margin=dict(l=220, r=100, t=90, b=50),
    xaxis=dict(title=dict(text="Incremental units (total)", font=dict(size=12, color="#475569")),
               gridcolor="#f1f5f9", tickfont=dict(size=10, color="#64748b")),
    yaxis=dict(tickfont=dict(size=10.5, color="#334155")),
    plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
    hoverlabel=dict(bgcolor="white", bordercolor="#cbd5e1", font_size=12),
    bargap=0.25,
)
fig.show()


In [16]:
# Bubble chart: promotion lift vs price sensitivity — bubble size = average weekly sales volume
df_sc = res_df.merge(sku_meta[["sku", "functionality"]], on="sku", how="left")
df_sc["sig"] = df_sc["feat_pval"].apply(lambda p: "p<0.05" if p < 0.05 else ("p<0.10" if p < 0.10 else "p≥0.10"))

fill_map  = {"p<0.05": "#059669", "p<0.10": "#f59e0b", "p≥0.10": "#cbd5e1"}
edge_map  = {"p<0.05": "#047857", "p<0.10": "#d97706", "p≥0.10": "#94a3b8"}
max_sales = df_sc["avg_sales"].max()

fig = go.Figure()
for sig_level in ["p≥0.10", "p<0.10", "p<0.05"]:
    sub = df_sc[df_sc["sig"] == sig_level]
    if len(sub) == 0:
        continue
    sizes = np.clip(sub["avg_sales"] / max_sales * 32 + 8, 10, 42)
    fig.add_trace(go.Scatter(
        x=sub["price_elasticity"], y=sub["pct_lift"], mode="markers",
        marker=dict(size=sizes, color=fill_map[sig_level],
                    line=dict(width=1.5, color=edge_map[sig_level]), opacity=0.85),
        name=sig_level,
        customdata=np.column_stack([sub["sku"], sub["functionality"], sub["avg_sales"]]),
        hovertemplate="<b>SKU %{customdata[0]:.0f}</b> · %{customdata[1]}<br>"
                      "Price elasticity: %{x:.2f}<br>Promotion lift: %{y:.1f}%<br>"
                      "Avg sales: %{customdata[2]:.1f}/wk<extra></extra>",
    ))

fig.add_shape(type="line", x0=df_sc["price_elasticity"].min() - 0.2,
              x1=df_sc["price_elasticity"].max() + 0.2, y0=0, y1=0,
              line=dict(color="#94a3b8", dash="dash", width=1))
fig.add_shape(type="line", x0=0, x1=0,
              y0=min(df_sc["pct_lift"].min() - 30, -30),
              y1=df_sc["pct_lift"].max() + 50,
              line=dict(color="#94a3b8", dash="dash", width=1))

for text, xf, yf in [
    ("High lift · Price inelastic", 0.02, 0.98),
    ("High lift · Price elastic",   0.98, 0.98),
    ("Low lift · Price inelastic",  0.02, 0.02),
    ("Low lift · Price elastic",    0.98, 0.02),
]:
    fig.add_annotation(
        text=f"<i>{text}</i>", xref="paper", yref="paper", x=xf, y=yf,
        showarrow=False, font=dict(size=9.5, color="#94a3b8"),
        xanchor="left" if xf < 0.5 else "right",
        yanchor="top" if yf > 0.5 else "bottom")

fig.update_layout(
    title=dict(
        text="<b>Promotion Lift vs Price Elasticity</b><br>"
             "<span style='font-size:12px;color:#64748b'>Bubble size = average weekly sales</span>",
        font=dict(size=16, color="#0f172a"), x=0.01, xanchor="left"),
    height=520, margin=dict(l=60, r=60, t=90, b=60),
    xaxis=dict(title=dict(text="Price elasticity", font=dict(size=12, color="#475569")),
               gridcolor="#f1f5f9", zeroline=False, tickfont=dict(size=10, color="#64748b")),
    yaxis=dict(title=dict(text="Promotion lift (%)", font=dict(size=12, color="#475569")),
               gridcolor="#f1f5f9", zeroline=False, tickfont=dict(size=10, color="#64748b")),
    plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
    legend=dict(title=dict(text="Significance", font=dict(size=11, color="#475569")),
                bgcolor="rgba(255,255,255,0.9)", bordercolor="#e2e8f0", borderwidth=1),
    hoverlabel=dict(bgcolor="white", bordercolor="#cbd5e1", font_size=12),
)
fig.show()


## All-SKU results table

In [17]:
# Full results table for all 44 SKUs: lift %, confidence interval, p-value, incremental units, and model fit
tbl = res_df.merge(sku_meta[["sku", "functionality"]], on="sku", how="left")
tbl = tbl[["sku", "functionality", "promo_weeks", "avg_sales_off", "avg_sales_on",
           "pct_lift", "lift_ci_lo", "lift_ci_hi", "feat_pval", "incr_per_week",
           "total_incremental", "price_elasticity", "r_squared"]].copy()
tbl.columns = ["SKU", "Category", "Promo wks", "Sales (off)", "Sales (on)",
               "Lift %", "CI lo %", "CI hi %", "p-value", "Incr/wk",
               "Total incr", "Price elast", "R²"]

for c in ["Lift %", "CI lo %", "CI hi %", "Price elast"]:
    tbl[c] = tbl[c].round(1)
tbl["Sales (off)"] = tbl["Sales (off)"].round(1)
tbl["Sales (on)"] = tbl["Sales (on)"].round(1)
tbl["Incr/wk"] = tbl["Incr/wk"].round(1)
tbl["Total incr"] = tbl["Total incr"].round(0).astype(int)
tbl["R²"] = tbl["R²"].round(3)
tbl["p-value"] = tbl["p-value"].round(4)

display(tbl.sort_values("Lift %", ascending=False).style.format({"p-value": "{:.4f}"})
        .background_gradient(subset=["Lift %"], cmap="Greens")
        .background_gradient(subset=["R²"], cmap="Blues"))


,SKU,Category,Promo wks,Sales (off),Sales (on),Lift %,CI lo %,CI hi %,p-value,Incr/wk,Total incr,Price elast,R²
10,11,01.Streaming sticks,1,56.900000,114.000000,232.000000,-29.400000,1460.000000,0.1325,131.900000,132,-2.300000,0.479000
5,6,04.Selfie sticks,10,13.100000,46.800000,215.300000,122.400000,347.100000,0.0000,28.300000,283,-0.800000,0.623000
37,38,06.Mobile phone accessories,8,7.200000,36.900000,177.600000,69.300000,355.200000,0.0001,12.800000,102,-0.800000,0.636000
11,12,01.Streaming sticks,21,22.100000,106.500000,88.900000,35.200000,163.900000,0.0004,19.700000,413,-1.800000,0.662000
28,29,06.Mobile phone accessories,10,35.400000,97.000000,78.300000,3.700000,206.900000,0.0398,27.700000,277,-1.800000,0.545000
20,21,04.Selfie sticks,17,13.200000,40.500000,72.500000,5.600000,181.600000,0.0323,9.600000,163,-1.100000,0.560000
42,43,11.Fitness trackers,90,5.100000,11.400000,66.900000,8.000000,158.000000,0.0236,3.400000,310,-1.500000,0.489000
16,17,06.Mobile phone accessories,44,16.400000,41.900000,61.900000,21.900000,115.100000,0.0013,10.100000,446,-1.600000,0.642000
0,1,06.Mobile phone accessories,19,11.200000,57.600000,52.500000,9.800000,111.700000,0.0137,5.900000,112,-1.300000,0.764000
12,13,09.Smartphone stands,9,15.600000,31.100000,50.000000,6.000000,112.300000,0.0248,7.800000,70,-1.200000,0.426000


## SKU deep-dive
Select a SKU for detailed promotion analysis.

In [ ]:
# Interactive deep-dive: pick a SKU to see its sales timeline, promo vs non-promo boxplot, and model coefficients

sku_list_dd = sorted(res_df["sku"].unique())
sku_labels_dd = {s: f"SKU {s} — {sku_meta[sku_meta.sku==s].functionality.values[0]}" for s in sku_list_dd}

try:
    _promo_output.close()
except NameError:
    pass

_promo_sku = widgets.Dropdown(options=[(sku_labels_dd[s], s) for s in sku_list_dd], value=sku_list_dd[0], description="SKU:")
_promo_btn = widgets.Button(description="Analyse", button_style="success", icon="search")
_promo_output = widgets.Output()

def _run_deepdive(_):
    _promo_output.clear_output()
    with _promo_output:
        sku_id = _promo_sku.value
        row = res_df[res_df["sku"] == sku_id].iloc[0]

        sig_text = "✓ Significant" if row.feat_pval < 0.05 else ("~ Marginal" if row.feat_pval < 0.10 else "✗ Not significant")
        display(metric_row([
            metric_card("Promotion lift", f"{row.pct_lift:.1f}%", f"CI: [{row.lift_ci_lo:.1f}%, {row.lift_ci_hi:.1f}%]"),
            metric_card("Incr / promo week", f"{row.incr_per_week:.1f}", "units"),
            metric_card("Total incremental", f"{row.total_incremental:,.0f}", f"over {int(row.promo_weeks)} promo weeks"),
            metric_card("Significance", sig_text, f"p = {row.feat_pval:.4f}"),
        ]))

        # Sales over time — green shading marks the weeks when the SKU was promoted
        df_raw = raw[raw["sku"] == sku_id].sort_values("week")
        fig = go.Figure()
        for w in df_raw[df_raw["feat_main_page"] == 1]["week"]:
            fig.add_vrect(x0=w - pd.Timedelta(days=3), x1=w + pd.Timedelta(days=3),
                          fillcolor="rgba(16,185,129,0.12)", line_width=0)
        fig.add_trace(go.Scatter(x=df_raw["week"], y=df_raw["weekly_sales"], mode="lines",
                                 line=dict(color="#0f172a", width=1.8), name="Sales"))
        promo_df = df_raw[df_raw["feat_main_page"] == 1]
        fig.add_trace(go.Scatter(x=promo_df["week"], y=promo_df["weekly_sales"], mode="markers",
                                 marker=dict(size=7, color=C_GREEN, symbol="diamond"), name="Promoted"))
        fig.update_layout(title=f"SKU {sku_id} — Sales with promotion periods highlighted",
                          height=350, xaxis_title="Week", yaxis_title="Weekly sales",
                          legend=dict(orientation="h", y=-0.15))
        fig.show()

        # Side-by-side comparison: sales distribution when featured vs not featured
        df_box = df_raw.copy()
        df_box["Status"] = df_box["feat_main_page"].map({1: "Featured", 0: "Not featured"})
        fig2 = go.Figure()
        for lab, col in [("Not featured", "#cbd5e1"), ("Featured", C_GREEN)]:
            sub = df_box[df_box["Status"] == lab]
            fig2.add_trace(go.Box(y=sub["weekly_sales"], name=lab, marker_color=col, boxmean="sd"))
        fig2.update_layout(title=f"SKU {sku_id} — Sales distribution", height=320, yaxis_title="Weekly sales")
        fig2.show()

        # Model coefficients: how much each factor (price, promo, trend) influences sales
        model_s, _ = run_scanpro_single(proc, sku_id)
        params = model_s.params.drop("const", errors="ignore")
        pvals = model_s.pvalues.drop("const", errors="ignore")
        keep = [c for c in params.index if not c.startswith("month_")]
        params, pvals = params[keep], pvals[keep]
        colors_c = [C_GREEN if p < 0.05 else ("#fbbf24" if p < 0.10 else "#cbd5e1") for p in pvals]

        fig3 = go.Figure(go.Bar(y=params.index, x=params.values, orientation="h", marker_color=colors_c,
                                text=[f"{v:.3f}" for v in params.values], textposition="outside"))
        fig3.add_vline(x=0, line_color="#94a3b8", line_width=1)
        fig3.update_layout(title="SCAN*PRO coefficients (excl. month dummies)", height=250,
                           margin=dict(l=140, r=60, t=40, b=20), xaxis_title="Coefficient")
        fig3.show()

        # Expandable section with the full statistical regression output for technical review
        display(HTML(f"<details><summary><b>Full regression output</b></summary><pre>{model_s.summary().as_text()}</pre></details>"))

_promo_btn.on_click(_run_deepdive)
display(widgets.VBox([
    widgets.HBox([_promo_sku, _promo_btn]),
    _promo_output,
]))


Output()

## Methodology

### Model choice

The task is to measure the **causal effect** of featuring a product on the platform's main page on its weekly sales. This is fundamentally a coefficient estimation problem, not a prediction problem — we need an interpretable model that isolates the promotion effect from confounders (price, seasonality, trend).

We use the **SCAN\*PRO model** (Van Heerde et al., 2002), the standard marketing science framework for promotion effectiveness measurement. It is estimated per SKU via **OLS** because:
1. OLS produces unbiased coefficient estimates with valid standard errors for hypothesis testing
2. The model is fully interpretable — each coefficient has a direct economic meaning
3. With ~98 observations and ~15 parameters, OLS has adequate power and degrees of freedom
4. The goal is estimation of β₂ (promotion lift), not out-of-sample prediction

### Specification

```
ln(sales_it) = α + β₁·ln(price_it) + β₂·feat_it + β₃·trend_t 
             + β₄·ln(price_i,t-1) + β₅·ln(price_i,t-2) + Σ γ_k·month_k + ε_it
```

| Parameter | Interpretation | Why included |
|-----------|---------------|-------------|
| β₁ | Price elasticity | Separates price discount effect from promotion effect |
| β₂ | Promotion coefficient | The variable of interest |
| β₃ | Trend control | Prevents attributing demand growth to promotions |
| β₄, β₅ | Reference-price / stockpiling dynamics | Omitting these biases β₁ and β₂ (Van Heerde et al., 2002) |
| γ_k | Seasonal fixed effects | Prevents attributing seasonal peaks to promotions |

**Functional form:** Log-log for price (β₁ = elasticity directly). Linear for the binary promotion dummy (exp(β₂) − 1 = proportional lift). This is the standard SCAN\*PRO parameterisation.

### Metrics

| Metric | Formula |
|--------|--------|
| Lift % | (exp(β₂) − 1) × 100 |
| Incremental/week | avg baseline sales × (exp(β₂) − 1) |
| Total incremental | incremental/week × promoted weeks |
| 95% CI | exp(β₂ ± 1.96·SE) − 1 |
| Significance | OLS t-test on β₂, p < 0.05 |

### Assumptions and limitations

- Log-linear price-demand relationship (constant elasticity)
- Zero-sales weeks excluded (log(0) is undefined)
- Same-week promotion effect only (no lagged promotion carryover)
- No cross-SKU competitive effects (promotion of SKU A doesn't cannibalise SKU B)
- Promotions are treated as exogenous — in reality, retailers may choose to promote SKUs they expect to sell well (selection bias). Without an instrument or natural experiment, we cannot fully rule this out.

### References

- Van Heerde, H.J., Leeflang, P.S.H., & Wittink, D.R. (2002). How promotions work: SCAN\*PRO-based evolutionary model building. *Schmalenbach Business Review*, 54, 198–220.
- Hanssens, D.M., Parsons, L.J., & Schultz, R.L. (2001). *Market Response Models*, 2nd edition. Kluwer.
- Van Heerde, H.J., Leeflang, P.S.H., & Wittink, D.R. (2004). Decomposing the Sales Promotion Bump. *Marketing Science*, 23(3), 317–334.